# Gaussian fit to identify reporter positive cells

In this notebook, we identify reporter positive cells in the global dataset (including both neurons and non-neuronal cells). *EGFP* marks colon-innervating neurons and *dTomato* marks bladder-innervating neurons, both via AAV retrograde labeling; dual-positive cells innervate both organs. We fit a Gaussian to the background log-expression distribution of each reporter and classify cells above the chosen percentile threshold as reporter-positive. Results are saved in `reporter_classes_gaussian_thresholds.csv`, which we can use to add reporter negative/positive information to adata objects for downstream analysis.

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

In [1]:
from pathlib import Path
import os
import sys
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Setup

In [2]:
sys.path.append(str(Path.cwd().resolve().parents[1]))

from config.paths import BASE_DIR

adata_path = BASE_DIR / "data/h5ad/export_03/03b_seurat/adata-merged-labels.h5ad"
adata_480_path = BASE_DIR / "data/h5ad/export_02/02b_leiden/adata-leiden-480.h5ad"
output_dir = BASE_DIR / "data/h5ad/export_03/03b_thresholds"
output_dir.mkdir(parents=True, exist_ok=True)

plot_out_dir = BASE_DIR / "figures" / "plots"
plot_out_dir.mkdir(parents=True, exist_ok=True)


### Read and process adata


In [3]:
# global dataset, including reporter genes
adata_480 = sc.read_h5ad(adata_480_path)

# labeled dataset, reporter genes removed
adata = sc.read_h5ad(adata_path)

In [ ]:
# Xenium output appends a sample-index suffix (e.g. "aabc-1-2"); clip to the first two
# segments to match the barcode format used in adata and enable row alignment.
index_clipped = ["-".join(barcode.split('-')[0:2]) for barcode in adata_480.obs.index]
adata_480.obs.index = index_clipped

In [5]:
for adata_select in [adata, adata_480]:
    print(adata_select.shape)
    print(min(adata_select.obs['total_counts']))

(147036, 475)
51.0
(147036, 480)
51.0


## Gaussian threshold setting

In [6]:
from scipy.stats import norm

def to_dense(x):
    return x.toarray().ravel() if hasattr(x, "toarray") else np.ravel(x)

def gaussian_thresholds(adata_ref,
                  genes = ["EGFP_Seq1", "dTomato_Seq2"],
                  gene_thresholds = (0.99, 0.99),
                  highlight_barcodes=None,
                  highlight_color='orange',
                  highlight_size = 30,
                  save_path=None,
                  return_reporter_classes=False,
                 show_histograms=False):
    """
    Identify reporter-positive cells using Gaussian distribution thresholds.

    Models background expression as a Gaussian in log-space and classifies cells
    as positive if expression exceeds the specified percentile threshold.

    Parameters
    ----------
    adata_ref : AnnData
        AnnData with raw counts in .layers["counts"]
    genes : list of str
        Reporter gene names to analyze
    gene_thresholds : tuple of float
        Percentile thresholds (0-1) for each gene
    highlight_barcodes : array-like, optional
        Cell barcodes to highlight in plot
    highlight_color : str
        Color for highlighted cells
    highlight_size : int
        Point size for highlighted cells
    save_path : str, optional
        Path to save scatter plot
    return_reporter_classes : bool
        If True, returns DataFrame with reporter classifications
    show_histograms : bool
        If True, shows histogram of background distribution for each gene
    
    Returns
    -------
    pd.DataFrame or None
        DataFrame with columns 'sample_id', '{gene}_hi', 'EGFP_dTomato_dual_hi'
        if return_reporter_classes=True, else None.
    """

    # Work on copies to avoid unintended side effects
    adata_ref = adata_ref.copy()

    # Standardize gene names
    adata_ref.var_names = adata_ref.var_names.where(adata_ref.var_names != "tdTomato_Seq2", "dTomato_Seq2")

    # Ensure we are using raw counts for the math
    adata_ref.X = adata_ref.layers["counts"].copy()

    gauss_results = {}

    for i, g in enumerate(genes):
        print(g)
        threshold_percentile = gene_thresholds[i]

        # 1. Get full expression vector for this gene
        x_raw_full = to_dense(adata_ref[:, g].X)
        x_log_full = np.log1p(x_raw_full)

        # 2. INDEPENDENT Background Filter
        # We only use cells with >1 count in THIS gene to model the noise
        specific_bg = x_raw_full > 1

        # log distribution of cells that pass the gene specific background filter
        x_log_bg = x_log_full[specific_bg]

        # 3. Calculate Gaussian parameters from this gene's noise
        mu, sigma = np.mean(x_log_bg), np.std(x_log_bg)
        multiplier = norm.ppf(threshold_percentile)
        thr_log = mu + multiplier * sigma

        # 4. Apply threshold to ALL cells
        # Now a cell with 0 dTomato can still be EGFP_hi
        adata_ref.obs[f"{g}_hi"] = x_log_full > thr_log

        gauss_results[g] = {
            "full_log": x_log_full,
            "thr_log": thr_log
        }
        print(f"{g}: {threshold_percentile} percentile threshold set at {thr_log:.3f} (Z={multiplier:.3f})")

        if show_histograms:
            plt.figure(figsize=(4, 3))
            # Plot the background distribution used for the model
            plt.hist(x_log_bg, bins=100, color='gray', alpha=0.7)
            # Add the threshold line
            plt.axvline(thr_log, color='red', linestyle='--', label=f'Threshold: {thr_log:.2f}')
            plt.title(f"Background Distribution: {g}")
            plt.xlabel("log1p Expression")
            plt.ylabel("Frequency")
            plt.ylim(0,100)
            plt.legend()
            sns.despine()
            plt.show()  # This ensures the plot renders before the next loop iteration


    # Identify Dual-high (Logical AND of independent results)
    adata_ref.obs["EGFP_dTomato_dual_hi"] = (
        adata_ref.obs["EGFP_Seq1_hi"] & adata_ref.obs["dTomato_Seq2_hi"]
    )

    # --- Highlight Logic ---
    is_highlight = adata_ref.obs_names.isin(highlight_barcodes) if highlight_barcodes is not None else np.zeros(adata_ref.n_obs, dtype=bool)

    # Define plotting groups
    dual_mask = adata_ref.obs["EGFP_dTomato_dual_hi"].values
    final_dual = dual_mask & ~is_highlight
    final_highlight = is_highlight

    # --- Plotting ---
    x_egfp = gauss_results["EGFP_Seq1"]["full_log"]
    x_td = gauss_results["dTomato_Seq2"]["full_log"]

    sns.set_theme(style="ticks")
    plt.figure(figsize=(3, 3), dpi=300)
    ax = plt.gca()

    # 1. All cells (Background)
    ax.scatter(x_egfp, x_td, s=2, color="lightgray", alpha=0.1, label="Background", rasterized=True)

    # 2. Dual Positive cells (Blue)
    if final_dual.any():
        ax.scatter(x_egfp[final_dual], x_td[final_dual],
                   # s=30, facecolors=(0.0, 0.47, 1.0, 0.4), edgecolors="blue",
                   s=30, facecolors=(0.0, 0.0, 1.0, 0.5), edgecolors="black",
                   linewidths=0.8, label="Dual Positive")

    # 3. Highlighted cells (Orange/Custom) - Drawn last with high zorder
    if final_highlight.any():
        print(f"number of barcodes to highlight: {sum(final_highlight)}")
        ax.scatter(x_egfp[final_highlight], x_td[final_highlight],
                   s=highlight_size, facecolors=highlight_color,
                   alpha=1.0, label="Highlighted", zorder=10)

    # Threshold lines
    ax.axvline(gauss_results["EGFP_Seq1"]["thr_log"], color="red", ls="--", lw=1, alpha=0.8)
    ax.axhline(gauss_results["dTomato_Seq2"]["thr_log"], color="red", ls="--", lw=1, alpha=0.8)

    ax.set_xlabel("log1p EGFP_Seq1")
    ax.set_ylabel("log1p dTomato_Seq2")
    sns.despine()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()

    if return_reporter_classes:
        return adata_ref.obs[['sample_id', 'EGFP_Seq1_hi', 'dTomato_Seq2_hi', 'EGFP_dTomato_dual_hi']]

In [ ]:
# Thresholds for each reporter were set to remove false positives and prevent false
# negatives by manual inspection of high confidence reporter-positive or
# reporter-negative cells in Xenium Explorer.
reporter_classes_updated_dual = gaussian_thresholds(adata_480, 
                                            genes = ["EGFP_Seq1", "dTomato_Seq2"],
                                            gene_thresholds = (0.9998, 0.99),
                                            return_reporter_classes = True,
                                            save_path = os.path.join(plot_out_dir, "gaussian_thresholds_on_global_data.pdf")
                                             )

# add reporter status, either single positive or double postive or negative
reporter_classes_updated_dual['reporter_status'] = 'Negative'
reporter_classes_updated_dual.loc[reporter_classes_updated_dual['EGFP_Seq1_hi']==True, 'reporter_status'] = 'EGFP_Seq1_hi'
reporter_classes_updated_dual.loc[reporter_classes_updated_dual['dTomato_Seq2_hi']==True, 'reporter_status'] = 'dTomato_Seq2_hi'
reporter_classes_updated_dual.loc[reporter_classes_updated_dual['EGFP_dTomato_dual_hi']==True, 'reporter_status'] = 'EGFP_dTomato_dual_hi'


print(sum(reporter_classes_updated_dual['EGFP_Seq1_hi']))
print(sum(reporter_classes_updated_dual['dTomato_Seq2_hi']))
print(sum(reporter_classes_updated_dual['EGFP_dTomato_dual_hi']))

In [9]:
reporter_classes_updated_dual[reporter_classes_updated_dual['EGFP_Seq1_hi']==True]

,sample_id,EGFP_Seq1_hi,dTomato_Seq2_hi,EGFP_dTomato_dual_hi,reporter_status
aejfpdmp-1,TMA00304,True,False,False,EGFP_Seq1_hi
agcgokgc-1,TMA00304,True,False,False,EGFP_Seq1_hi
ahgpcegf-1,TMA00304,True,False,False,EGFP_Seq1_hi
ahlhenfj-1,TMA00304,True,False,False,EGFP_Seq1_hi
aogalboa-1,TMA00304,True,False,False,EGFP_Seq1_hi
...,...,...,...,...,...
dhfhkobe-1,TMA00314,True,False,False,EGFP_Seq1_hi
dhgckklb-1,TMA00314,True,False,False,EGFP_Seq1_hi
dhjpmlak-1,TMA00314,True,False,False,EGFP_Seq1_hi
kchcnchj-1,TMA00314,True,False,False,EGFP_Seq1_hi


## Save table of reporter assignments per cell barcode

In [10]:
reporter_classes_updated_dual.to_csv(os.path.join(output_dir, 'reporter_classes_gaussian_thresholds.csv'))